In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn.linear_model as lm
import pandas as pd
from data_tools import load_data

import pickle

import mat73

In [ ]:
features = load_data('ESR1_data_all2.mat')
power = features[0]
coherence = features[1]
granger = features[2]

## Project into the global model

In [ ]:
import sys
sys.path.append('/home/austin/Aggression/Code/NMF')
from nmf_elastic import NMF_logistic



In [ ]:
# Process data identically
granger = np.exp(granger)
granger[granger>10] = 10
power = power*10
power[power>6] = 6

X = np.hstack((power,coherence,granger))

In [ ]:
nFact = 8
myDict = {}

nIter = 20000
model = NMF_logistic(nFact,nIter=nIter,LR=1e-3,mu=1.0,batchSize=100)

my_dict = pickle.load(open('../Unbalanced_Elastic_12_enc_1.0.p','rb'))
model.A_enc = my_dict['A_enc']
model.B_enc = my_dict['B_enc']

In [ ]:
S_test = model.transform(X)

In [ ]:
np.savetxt('ESR1_Global_model_scores.csv',S_test,delimiter=',',fmt='%0.8f')

## Project into  the individual models

In [ ]:
import cloudpickle
model_list = cloudpickle.load(open('../SingleRegionModels/AggresionPro.p','rb'))

In [ ]:
model_list.keys()

In [ ]:
for i in range(11):
    S_test_single = model_list['models'][i].transform(power[:,i*56:(i+1)*56])
    sname = 'Single_region_esr_region_%d.csv'%int(i)
    np.savetxt(sname,S_test_single,delimiter=',',fmt='%0.8f')